# Práctica: construyendo un Transformer desde cero

**Duración estimada:** 2 horas · **Nivel:** licenciatura · **Herramientas:** Python + PyTorch (Google Colab con CPU es suficiente)

En esta práctica vas a implementar, pieza por pieza, los componentes de la arquitectura Transformer (Vaswani et al., 2017, *Attention Is All You Need*) y al final entrenarás un pequeño modelo que aprende a **ordenar secuencias de dígitos**. Después abrirás el modelo para ver *a qué presta atención*.

### Objetivos de aprendizaje
1. Explicar por qué el Transformer necesita información de posición y cómo la aporta la codificación sinusoidal.
2. Implementar la atención de producto punto escalada, con y sin máscara causal.
3. Implementar la atención multi-cabeza y verificarla contra la implementación de PyTorch.
4. Ensamblar un bloque Transformer (atención + red feed-forward + conexiones residuales + LayerNorm).
5. Entrenar un mini-Transformer, interpretar sus mapas de atención y realizar un experimento de ablación.

### Agenda sugerida
| Parte | Tema | Tiempo |
|---|---|---|
| 0 | Preparación del entorno | 5 min |
| 1 | Codificación posicional | 15 min |
| 2 | Atención de producto punto escalada | 25 min |
| 3 | Atención multi-cabeza | 20 min |
| 4 | Bloque Transformer | 15 min |
| 5 | Entrenar un mini-Transformer | 25 min |
| 6 | Interpretación y ablación | 10 min |
| 7 | Preguntas de cierre | 5 min |

### Instrucciones
- Busca las celdas marcadas con **`# TODO`** y complétalas. No modifiques las celdas de verificación (✅): sirven para comprobar tu trabajo.
- Responde las preguntas marcadas con ✍️ en la celda de texto que aparece debajo de cada una.
- Ejecuta las celdas **en orden**. Si algo falla, revisa primero las celdas anteriores.

## Parte 0 · Preparación (5 min)

In [ ]:
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch", torch.__version__, "| dispositivo:", device)

## Parte 1 · Codificación posicional (15 min)

A diferencia de una RNN, el Transformer procesa todos los tokens **en paralelo** y la atención trata la entrada como un *conjunto*: si permutas los tokens, la salida se permuta de la misma forma. Por eso hay que **inyectar la posición** de cada token. El artículo original suma a cada embedding un vector fijo:

$$
PE_{(pos,\,2i)} = \sin\!\left(\frac{pos}{10000^{2i/d_{model}}}\right), \qquad
PE_{(pos,\,2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d_{model}}}\right)
$$

Cada par de dimensiones $(2i, 2i+1)$ oscila con una frecuencia distinta: las primeras dimensiones cambian rápido con la posición y las últimas, muy lento.

### Ejercicio 1
Implementa `codificacion_posicional(max_len, d_model)`, que devuelve un tensor de forma `(max_len, d_model)`.

*Pista:* construye `pos` con forma `(max_len, 1)` y un vector de frecuencias de forma `(d_model/2,)`; el producto se expande por *broadcasting*. Usa `pe[:, 0::2]` y `pe[:, 1::2]` para llenar columnas pares e impares.

In [ ]:
def codificacion_posicional(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    # TODO: llena las columnas pares con senos y las impares con cosenos
    return pe

In [ ]:
# ✅ Verificación
pe = codificacion_posicional(50, 16)
assert pe.shape == (50, 16), "La forma debe ser (max_len, d_model)"
assert torch.allclose(pe[0, 0::2], torch.zeros(8)), "En pos=0 los senos valen 0"
assert torch.allclose(pe[0, 1::2], torch.ones(8)), "En pos=0 los cosenos valen 1"
assert abs(pe[1, 0].item() - math.sin(1.0)) < 1e-6, "Revisa la frecuencia de la primera dimensión"
assert abs(pe[3, 15].item() - math.cos(3 / 10000 ** (14 / 16))) < 1e-6, "Revisa el exponente 2i/d_model"
print("¡Ejercicio 1 correcto!")

plt.figure(figsize=(8, 3.5))
plt.imshow(codificacion_posicional(100, 64).T, aspect="auto", cmap="RdBu")
plt.xlabel("posición"); plt.ylabel("dimensión"); plt.colorbar()
plt.title("Codificación posicional sinusoidal (d_model = 64)")
plt.show()

✍️ **Pregunta 1.** Observa la figura. ¿Por qué conviene usar varias frecuencias y no una sola? ¿Qué pasaría si solo usáramos la dimensión 0 (un único seno)?

*Tu respuesta:*

## Parte 2 · Atención de producto punto escalada (25 min)

Cada token produce tres vectores: una **consulta** $q$ ("¿qué busco?"), una **clave** $k$ ("¿qué ofrezco?") y un **valor** $v$ ("¿qué información aporto?"). La salida para cada consulta es un promedio ponderado de los valores, donde los pesos miden la similitud consulta–clave:

$$
\text{Atención}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V
$$

Con una **máscara** se impide que ciertas posiciones se vean: antes del softmax se pone $-\infty$ en las posiciones prohibidas, de modo que su peso queda en 0.

### Ejercicio 2
Implementa `atencion(Q, K, V, mascara=None)`. Debe funcionar con cualquier número de dimensiones iniciales (por ejemplo `(lote, n, d_k)` o `(lote, cabezas, n, d_k)`), así que opera siempre sobre las **dos últimas** dimensiones. Devuelve la salida y la matriz de pesos.

Convención: `mascara` es un tensor booleano donde `True` significa *"esta posición puede verse"*.

*Pistas:* `K.transpose(-2, -1)`, `tensor.masked_fill(condicion, valor)`, `torch.softmax(x, dim=-1)`.

In [ ]:
def atencion(Q, K, V, mascara=None):
    d_k = Q.size(-1)
    # TODO 1: calcula los puntajes Q·Kᵀ / √d_k
    # TODO 2: si hay máscara, pon -inf donde mascara sea False
    # TODO 3: aplica softmax sobre la última dimensión para obtener los pesos
    # TODO 4: multiplica los pesos por V
    pesos = None
    salida = None
    return salida, pesos

In [ ]:
# ✅ Verificación: comparamos contra la implementación oficial de PyTorch
Q, K, V = torch.randn(2, 5, 8), torch.randn(2, 5, 8), torch.randn(2, 5, 8)
salida, pesos = atencion(Q, K, V)
assert salida.shape == (2, 5, 8) and pesos.shape == (2, 5, 5), "Revisa las formas"
assert torch.allclose(pesos.sum(-1), torch.ones(2, 5)), "Cada fila de pesos debe sumar 1"
assert torch.allclose(salida, F.scaled_dot_product_attention(Q, K, V), atol=1e-5), "No coincide con PyTorch"
print("¡Ejercicio 2 correcto!")

### Ejercicio 3 · Máscara causal
En un modelo generativo (como GPT) el token en la posición $i$ **no puede ver el futuro**: solo las posiciones $j \le i$. Implementa `mascara_causal(n)`, que devuelve una matriz booleana `(n, n)` con `True` en la diagonal y por debajo de ella.

*Pista:* `torch.tril`.

In [ ]:
def mascara_causal(n):
    # TODO
    return None

In [ ]:
# ✅ Verificación
m = mascara_causal(5)
print(m.int())
salida, pesos = atencion(Q, K, V, m)
assert torch.all(pesos[:, ~m] == 0), "Hay atención hacia el futuro"
ref = F.scaled_dot_product_attention(Q, K, V, is_causal=True)
assert torch.allclose(salida, ref, atol=1e-5), "No coincide con PyTorch (is_causal=True)"
print("¡Ejercicio 3 correcto!")

fig, ejes = plt.subplots(1, 2, figsize=(8, 3.5))
for ax, (titulo, p) in zip(ejes, [("Sin máscara", atencion(Q, K, V)[1][0]), ("Máscara causal", pesos[0])]):
    ax.imshow(p.detach(), cmap="viridis", vmin=0, vmax=1)
    ax.set_title(titulo); ax.set_xlabel("clave (posición vista)"); ax.set_ylabel("consulta")
plt.tight_layout(); plt.show()

### Experimento · ¿Para qué sirve dividir entre $\sqrt{d_k}$?
Si las entradas de $q$ y $k$ tienen media 0 y varianza 1, el producto punto $q\cdot k$ tiene varianza $d_k$. Ejecuta la celda y observa qué le pasa al softmax cuando $d_k$ crece **sin** escalar. Usamos la *entropía* de cada fila de pesos: es alta si la atención se reparte y cercana a 0 si se concentra en un solo token.

In [ ]:
def entropia_media(pesos):
    return -(pesos * torch.log(pesos + 1e-12)).sum(-1).mean().item()

n = 16
print(f"Entropía máxima posible (atención uniforme sobre {n} tokens): {math.log(n):.2f}\n")
print(f"{'d_k':>6} | {'con escala':>10} | {'sin escala':>10} | {'peso máx. sin escala':>20}")
for d_k in [4, 64, 512, 4096]:
    q, k = torch.randn(1000, n, d_k), torch.randn(1000, n, d_k)
    s = q @ k.transpose(-2, -1)
    con, sin = torch.softmax(s / math.sqrt(d_k), -1), torch.softmax(s, -1)
    print(f"{d_k:>6} | {entropia_media(con):>10.2f} | {entropia_media(sin):>10.2f} | {sin.max(-1).values.mean().item():>20.3f}")

✍️ **Pregunta 2.** Según la tabla, ¿qué le ocurre a la atención sin escalar cuando $d_k$ es grande? ¿Por qué eso dificulta el entrenamiento? *(Piensa en el gradiente del softmax cuando una salida es casi 1 y las demás casi 0.)*

*Tu respuesta:*

## Parte 3 · Atención multi-cabeza (20 min)

En lugar de una sola atención sobre vectores de dimensión $d_{model}$, el Transformer usa $h$ **cabezas** en paralelo, cada una sobre una proyección de dimensión $d_k = d_{model}/h$. Cada cabeza puede especializarse en un tipo de relación distinto. Al final se concatenan y se mezclan con una proyección $W_O$:

$$
\text{MultiHead}(X) = \text{Concat}(\text{cabeza}_1, \dots, \text{cabeza}_h)\,W_O, \qquad \text{cabeza}_i = \text{Atención}(XW_Q^{i}, XW_K^{i}, XW_V^{i})
$$

En la práctica, las $h$ proyecciones se calculan con **una sola** capa lineal de $d_{model}\to d_{model}$ y luego se "parte" el resultado en cabezas.

### Ejercicio 4
Completa el método `forward`. Las formas que debes recorrer son:

`(B, N, d_model)` → proyección → `(B, N, h, d_k)` → `transpose(1, 2)` → `(B, h, N, d_k)` → atención → `(B, h, N, d_k)` → `transpose(1, 2)` → `(B, N, h, d_k)` → `reshape` → `(B, N, d_model)` → $W_O$

Guarda los pesos de atención en `self.pesos` (los usaremos para visualizar).

In [ ]:
class AtencionMultiCabeza(nn.Module):
    def __init__(self, d_model, n_cabezas):
        super().__init__()
        assert d_model % n_cabezas == 0, "d_model debe ser divisible entre n_cabezas"
        self.h = n_cabezas
        self.d_k = d_model // n_cabezas
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.pesos = None

    def forward(self, x, mascara=None):
        B, N, d_model = x.shape
        # TODO 1: proyecta x con W_q, W_k, W_v y divide cada resultado en cabezas: (B, h, N, d_k)
        # TODO 2: aplica tu función atencion() y guarda los pesos en self.pesos
        # TODO 3: junta las cabezas de nuevo en (B, N, d_model) y aplica W_o
        return None

In [ ]:
# ✅ Verificación: copiamos nuestros pesos a nn.MultiheadAttention de PyTorch y comparamos salidas
torch.manual_seed(1)
d_model, h = 32, 4
mia = AtencionMultiCabeza(d_model, h)
oficial = nn.MultiheadAttention(d_model, h, batch_first=True)
with torch.no_grad():
    oficial.in_proj_weight.copy_(torch.cat([mia.W_q.weight, mia.W_k.weight, mia.W_v.weight]))
    oficial.in_proj_bias.copy_(torch.cat([mia.W_q.bias, mia.W_k.bias, mia.W_v.bias]))
    oficial.out_proj.weight.copy_(mia.W_o.weight)
    oficial.out_proj.bias.copy_(mia.W_o.bias)

x = torch.randn(3, 7, d_model)
y = mia(x)
assert y.shape == (3, 7, d_model), "La salida debe tener forma (B, N, d_model)"
assert mia.pesos.shape == (3, h, 7, 7), "self.pesos debe tener forma (B, h, N, N)"
y_ref, _ = oficial(x, x, x)
assert torch.allclose(y, y_ref, atol=1e-5), "La salida no coincide con nn.MultiheadAttention"
# con máscara causal (en PyTorch, attn_mask=True significa *bloquear*, al revés que nosotros)
y_c = mia(x, mascara_causal(7))
y_c_ref, _ = oficial(x, x, x, attn_mask=~mascara_causal(7))
assert torch.allclose(y_c, y_c_ref, atol=1e-5), "Falla con máscara causal"
print("¡Ejercicio 4 correcto!")
print("Número de parámetros:", sum(p.numel() for p in mia.parameters()))

✍️ **Pregunta 3.** El modelo de arriba tiene 4 cabezas. ¿Cuántos parámetros tendría con 1 cabeza? ¿Y con 8? Justifica con la fórmula del número de parámetros. ¿Qué ganamos entonces al usar varias cabezas?

*Tu respuesta:*

## Parte 4 · El bloque Transformer (15 min)

Un bloque combina dos subcapas, cada una envuelta en una **conexión residual** y una **normalización por capa** (LayerNorm):

1. Atención multi-cabeza (los tokens intercambian información entre sí).
2. Red *feed-forward* aplicada a cada posición por separado: `Linear(d_model→d_ff) → ReLU → Linear(d_ff→d_model)`.

Usaremos la variante **pre-LN** (normalizar *antes* de cada subcapa), que es la que usan hoy GPT y la mayoría de los modelos porque entrena de forma más estable que la versión original (*post-LN*):

```
x = x + Dropout(Atención(LayerNorm(x)))
x = x + Dropout(FFN(LayerNorm(x)))
```

### Ejercicio 5
Completa `forward` siguiendo exactamente las dos líneas de arriba.

In [ ]:
class BloqueTransformer(nn.Module):
    def __init__(self, d_model, n_cabezas, d_ff, dropout=0.1):
        super().__init__()
        self.atencion = AtencionMultiCabeza(d_model, n_cabezas)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mascara=None):
        # TODO: subcapa de atención con residual, luego subcapa feed-forward con residual
        return x

In [ ]:
# ✅ Verificación
torch.manual_seed(0)
bloque = BloqueTransformer(d_model=32, n_cabezas=4, d_ff=64).eval()
x = torch.randn(2, 6, 32)
y = bloque(x)
assert y.shape == x.shape, "El bloque debe conservar la forma de la entrada"
assert not torch.allclose(y, x), "El bloque no está modificando la entrada"
# Prueba de equivarianza: sin codificación posicional, permutar la entrada permuta la salida
perm = torch.randperm(6)
assert torch.allclose(bloque(x[:, perm]), y[:, perm], atol=1e-5), "Revisa el forward"
print("¡Ejercicio 5 correcto! Permutar la entrada simplemente permuta la salida.")

✍️ **Pregunta 4.** La última verificación muestra que el bloque es *equivariante a permutaciones*. Relaciona este resultado con la Parte 1: ¿por qué es indispensable la codificación posicional? Menciona además una ventaja de las conexiones residuales cuando apilamos muchos bloques.

*Tu respuesta:*

## Parte 5 · Entrenar un mini-Transformer (25 min)

**Tarea:** recibir 10 dígitos al azar y devolverlos **ordenados** de menor a mayor.

```
entrada:  3 9 1 4 1 5 9 2 6 5
salida:   1 1 2 3 4 5 5 6 9 9
```

Usaremos un Transformer tipo *encoder* (sin máscara causal): para cada posición de salida, el modelo predice qué dígito va ahí. Es una tarea pequeña pero no trivial: para saber qué dígito va en la posición 4, el modelo necesita "contar" cuántos dígitos menores hay en toda la secuencia.

### Ejercicio 6
Completa el `forward` del modelo:
1. Convierte los dígitos en vectores con `self.emb` y multiplícalos por $\sqrt{d_{model}}$ (así lo hace el artículo original, para que el embedding no quede opacado por la codificación posicional).
2. Si `self.usar_pe` es verdadero, suma `self.pe[:N]`.
3. Pasa el resultado por cada bloque de `self.bloques`.
4. Aplica `self.norm` y luego `self.salida` para obtener los *logits* de forma `(B, N, vocab)`.

In [ ]:
class MiniTransformer(nn.Module):
    def __init__(self, vocab=10, d_model=64, n_cabezas=4, n_capas=2, d_ff=128, max_len=32, usar_pe=True):
        super().__init__()
        self.d_model = d_model
        self.usar_pe = usar_pe
        self.emb = nn.Embedding(vocab, d_model)
        self.register_buffer("pe", codificacion_posicional(max_len, d_model))
        self.bloques = nn.ModuleList([BloqueTransformer(d_model, n_cabezas, d_ff, dropout=0.0) for _ in range(n_capas)])
        self.norm = nn.LayerNorm(d_model)
        self.salida = nn.Linear(d_model, vocab)

    def forward(self, x):            # x: (B, N) con enteros 0..9
        B, N = x.shape
        # TODO: pasos 1 a 4
        return None

In [ ]:
# ✅ Verificación rápida
modelo = MiniTransformer()
logits = modelo(torch.randint(0, 10, (4, 10)))
assert logits.shape == (4, 10, 10), "Los logits deben tener forma (B, N, vocab)"
print("¡Ejercicio 6 correcto! Parámetros del modelo:", sum(p.numel() for p in modelo.parameters()))

### Datos y entrenamiento
Los datos se generan al vuelo, así que nunca se repite exactamente el mismo ejemplo. No necesitas modificar estas celdas: léelas y asegúrate de entender cada línea. El entrenamiento tarda ~1–2 minutos en CPU.

In [ ]:
LONGITUD = 10

def generar_lote(tam_lote, n=LONGITUD):
    x = torch.randint(0, 10, (tam_lote, n))
    y = torch.sort(x, dim=1).values
    return x.to(device), y.to(device)

@torch.no_grad()
def evaluar(modelo, n=LONGITUD, tam=2000):
    modelo.eval()
    x, y = generar_lote(tam, n)
    pred = modelo(x).argmax(-1)
    modelo.train()
    exactitud_token = (pred == y).float().mean().item()
    exactitud_secuencia = (pred == y).all(dim=1).float().mean().item()
    return exactitud_token, exactitud_secuencia

def entrenar(modelo, pasos=2000, tam_lote=128, lr=1e-3):
    modelo.to(device).train()
    opt = torch.optim.AdamW(modelo.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=pasos)
    historial = []
    inicio = time.time()
    for paso in range(1, pasos + 1):
        x, y = generar_lote(tam_lote)
        logits = modelo(x)
        perdida = F.cross_entropy(logits.reshape(-1, 10), y.reshape(-1))
        opt.zero_grad()
        perdida.backward()
        opt.step()
        sched.step()
        historial.append(perdida.item())
        if paso % 250 == 0:
            tok, seq = evaluar(modelo)
            print(f"paso {paso:5d} | pérdida {perdida.item():.3f} | exactitud por dígito {tok:.1%} | secuencias perfectas {seq:.1%} | {time.time()-inicio:.0f}s")
    return historial

In [ ]:
torch.manual_seed(0)
modelo = MiniTransformer(usar_pe=True)
historial = entrenar(modelo)

plt.figure(figsize=(7, 3))
plt.plot(historial, alpha=0.4, label="por lote")
plt.plot(torch.tensor(historial).unfold(0, 50, 1).mean(1), label="promedio móvil (50)")
plt.yscale("log"); plt.xlabel("paso"); plt.ylabel("pérdida (entropía cruzada)"); plt.legend()
plt.title("Curva de entrenamiento"); plt.show()

In [ ]:
# Probemos el modelo con algunos ejemplos
x, y = generar_lote(5)
modelo.eval()
with torch.no_grad():
    pred = modelo(x).argmax(-1)
for xi, yi, pi in zip(x.tolist(), y.tolist(), pred.tolist()):
    marca = "✓" if yi == pi else "✗"
    print(f"{xi}  →  {pi}  {marca}")

✍️ **Pregunta 5.** ¿Qué exactitud por dígito y qué porcentaje de secuencias perfectas obtuviste? ¿Por qué el porcentaje de secuencias perfectas siempre es menor (y más exigente) que la exactitud por dígito?

*Tu respuesta:*

## Parte 6 · ¿A qué presta atención el modelo? Interpretación y ablación (10 min)

Como guardamos los pesos en `self.pesos`, podemos ver directamente qué posiciones de la entrada consulta cada posición de salida. Para que el patrón sea fácil de leer, usamos una entrada ya **ordenada al revés**.

In [ ]:
x = torch.tensor([[9, 8, 7, 6, 5, 4, 3, 2, 1, 0]], device=device)
modelo.eval()
with torch.no_grad():
    pred = modelo(x).argmax(-1)
print("entrada:   ", x[0].tolist())
print("predicción:", pred[0].tolist())

n_capas, n_cab = len(modelo.bloques), modelo.bloques[0].atencion.h
fig, ejes = plt.subplots(n_capas, n_cab, figsize=(3 * n_cab, 3 * n_capas))
etiquetas = [str(d) for d in x[0].tolist()]
for c, bloque in enumerate(modelo.bloques):
    for k in range(n_cab):
        ax = ejes[c, k]
        ax.imshow(bloque.atencion.pesos[0, k].cpu(), cmap="viridis", vmin=0, vmax=1)
        ax.set_title(f"capa {c+1} · cabeza {k+1}", fontsize=9)
        ax.set_xticks(range(10)); ax.set_xticklabels(etiquetas, fontsize=7)
        ax.set_yticks(range(10)); ax.set_yticklabels(range(10), fontsize=7)
fig.supxlabel("token de entrada al que se atiende (dígito)")
fig.supylabel("posición que consulta")
plt.tight_layout(); plt.show()

✍️ **Pregunta 6.** Describe al menos un patrón que veas en los mapas de atención (por ejemplo, ¿alguna cabeza de la última capa forma una diagonal invertida?, ¿alguna reparte la atención de forma uniforme?). Propón una interpretación: ¿qué información podría estar recolectando esa cabeza para resolver la tarea?

*Tu respuesta:*

### Experimento de ablación · ¿Y si quitamos la codificación posicional?
Entrena el mismo modelo con `usar_pe=False` y compara. **Antes de ejecutar**, escribe tu predicción en la Pregunta 7.

In [ ]:
torch.manual_seed(0)
modelo_sin_pe = MiniTransformer(usar_pe=False)
_ = entrenar(modelo_sin_pe)
print("\nCon codificación posicional:", "dígito {:.1%} | secuencia {:.1%}".format(*evaluar(modelo)))
print("Sin codificación posicional:", "dígito {:.1%} | secuencia {:.1%}".format(*evaluar(modelo_sin_pe)))

✍️ **Pregunta 7.** *(a) Antes de ejecutar:* ¿qué crees que pasará? *(b) Después:* ¿qué pasó y por qué? Usa el resultado de equivarianza de la Parte 4 en tu explicación. *(c)* ¿Por qué la exactitud por dígito sin codificación posicional no es 0% sino algo cercano al 20–30%?

*Tu respuesta:*

### Reto opcional · Generalización a longitudes nuevas
El modelo se entrenó solo con secuencias de 10 dígitos. Ejecuta la celda para evaluar con otras longitudes.

In [ ]:
for n in [5, 8, 10, 12, 16]:
    tok, seq = evaluar(modelo, n=n)
    print(f"longitud {n:2d}: exactitud por dígito {tok:6.1%} | secuencias perfectas {seq:6.1%}")

✍️ **Pregunta 8 (opcional).** ¿Generaliza el modelo a secuencias más largas que las de entrenamiento? ¿Y a más cortas? Propón una explicación y una modificación al entrenamiento que podría mejorar la generalización.

*Tu respuesta:*

## Parte 7 · Cierre (5 min)

✍️ **Preguntas finales** (respóndelas brevemente aquí o en tu reporte):

1. En una oración: ¿qué problema resuelve el mecanismo de atención respecto a una RNN?
2. ¿Cuál es el costo computacional de la atención respecto a la longitud $N$ de la secuencia? ¿Por qué esto es un problema para textos muy largos?
3. GPT usa bloques como los que construiste, pero con **máscara causal**. ¿Qué tendrías que cambiar en `MiniTransformer` para convertirlo en un modelo que genere texto carácter por carácter?

*Tus respuestas:*

### Para seguir aprendiendo
- Vaswani et al. (2017). *Attention Is All You Need*. arXiv:1706.03762.
- *The Illustrated Transformer*, de Jay Alammar (explicación visual muy recomendable).
- *The Annotated Transformer*, del grupo Harvard NLP (el artículo original implementado línea por línea en PyTorch).